# Study report: t2t-under-load

The study measures how the thread to thread wake latency (`t2t`) of
sem, mutex and msgq grows with periodic load timers and sleeper threads.
It tests a model of the slice reset: the p50 follows the number `n` of
timeouts that expire before the slice deadline, with one slope for the
three primitives. It also asks if the tail grows with the load.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-load-scaling@nucleo_g474re/st-t2t-under-load"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## The modes of a histogram

A mode is a group of bins that each hold at least 0.1 % of the samples,
with at most 2 cycles between neighbours. Mode 1 is the fastest.
`mode_share(k)` of the checks is the share of the samples in mode k
(`hypotheses.yaml`, `scripts/benchmarks/studylib/stats.py`). The mode
tables use the same rule (`nbfig.modes`), so they show what the checks
see. This rule reads the fine bins only. Where a table has the columns
"... with coarse bins", these columns use the same rule over the fine
and the coarse bins: a coarse bin of 32 cycles is one bin. A difference
between the two views is a mode that the checks do not see.

The app data keep only the bins with samples. In the histograms, the
line goes down to a floor at each end of a group of bins, and an empty
bin is a gap.

## The model

```text
t2t(load) = t2t_fast(no load) + c * n
```

- `n` is the number of timeouts that expire before the new slice
  deadline (20 ms after the reset). In each sample, the give resets the
  time slice of the taker (`update_cache()`). The reset inserts a
  timeout. The DLIST insert walks over these `n` timeouts with the
  timeout lock held.
- `c` is about 20..25 cycles per node (the wake data).
- Periodic timers (10..100 ms): `n` is about 0.47 x T. Sleepers
  (1..20 ms, all shorter than the slice): `n` = N.

Without load, two SysTick reprograms give the slow mode. When a timeout
comes before the slice deadline, the slice abort and add do not
reprogram SysTick. Without time slicing, the give inserts no slice
timeout (`study.md`).

The axis `load` of `thread-load-1m` has no numbers: its values are
timers and threads, and `n` is not a setting. The notebook takes `n` of
each load from `experiment.md` (table `N`).

In [ ]:
# n of each load: ex-thread-load-1m/experiment.md (axis load)
N = {"none": 0, "t16": 7.5, "t64": 30, "t256": 120, "sl8": 8, "sl64": 64}
df = inst.rows("thread-load-1m")
df["n"] = df["axis.load"].map(N)
t2t = df[df.metric == "t2t"].copy()
pd.Series(N, name="n").rename_axis("load").to_frame().T

## H1 a and H1 b: the modes without load and under load

H1 a: without load, the `t2t` has two modes, about 20 % fast and 80 %
slow, about 640 cycles apart (two SysTick reprograms). H1 b: with each
load, the `t2t` has one mode (fast mode > 95 %). Both checks are manual.
The column `step` is the median distance between neighbour modes.

In [ ]:
# a loaded point can have all its samples in the coarse bins: step and largest share use the coarse bins too
mt = nbfig.mode_table(inst, ["thread-load-1m"], "t2t", max_modes=2, coarse=True)
step = {}
largest = {}
for run, tree, point, rep in inst.points("thread-load-1m"):
    m = nbfig.modes(rep["hists"]["t2t"], coarse=True)
    step[point] = pd.Series([b[0] - a[0] for a, b in zip(m, m[1:])]).median() if len(m) > 1 else None
    largest[point] = round(max((s for _, _, s in m), default=0), 4)
mt["largest share"] = [largest[p] for _, p in mt.index]
mt["step"] = [step[p] for _, p in mt.index]
mt

In [ ]:
series = {p: h for p, h in nbfig.hists_of(inst, "thread-load-1m").items()
          if p.startswith("sem-") and not p.endswith("noslice")}
sem = df[(df.metric == "t2t") & df.point.isin(list(series))]
nbfig.hist(series, "Thread to thread per load: sem (up to the largest p99.9)", ["t2t"],
           xrange=(sem["min"].min() - 50, sem["p99.9"].max() + 50)).show()

## H1 c..g, i, j: the p50 against n

The checks c..g read the p50 of each load minus the p50 of `none` with
the same primitive. The no-load p50 is in the slow mode, and a loaded
point is in the fast mode. Thus the basis of the checks gives the range
`c x n - 640` (`hypotheses.yaml`). The grey bars are the ranges of the
checks. The two thin lines are the model with the lowest and the
highest `c`. The checks i and j read the control points without time
slicing (`noslice`) against `sem-none-noslice`. The x value of a control
point is the `n` of its load, also if the model gives no slice timeout
to it. The markers of the series have a small x offset, so that equal
values do not hide each other. The table below the figure gives the
ranges.

In [ ]:
import plotly.graph_objects as go

base = t2t[t2t["axis.load"] == "none"].set_index(["axis.primitive", "axis.slicing"]).p50
t2t["p50 - none"] = [r.p50 - base.get((r["axis.primitive"], r["axis.slicing"]), float("nan")) for _, r in t2t.iterrows()]
t2t["series"] = t2t["axis.primitive"] + [" noslice" if s == "noslice" else "" for s in t2t["axis.slicing"]]

# The ranges of the checks of H1 (expect diff_to): one per load, from hypotheses.yaml
lim = []
for c in inst.design_hypotheses["H1"]["checks"]:
    d = c.get("expect", {}).get("diff_to")
    if d:
        sel = c["select"]
        load = sel.get("axes", {}).get("load") or next(
            (r["axis.load"] for _, r in t2t.iterrows() if r.point == sel.get("point")), None)
        lim.append({"check": f"H1 {c['id']}", "load": load, "n": N.get(load), "lo": d["range"][0], "hi": d["range"][1],
                    "control": "point" in sel})
lim = pd.DataFrame(lim)

fig = go.Figure()
for k, (ctrl, name) in enumerate(((False, "range of H1 c..g"), (True, "range of H1 i, j"))):
    sub = lim[lim.control == ctrl]
    fig.add_trace(go.Scatter(x=sub.n, y=(sub.lo + sub.hi) / 2, mode="markers", name=name,
                             marker={"symbol": "line-ew-open", "size": 16, "color": "#6e6d66" if not ctrl else "#a3a29b",
                                     "line": {"width": 2, "color": "#6e6d66" if not ctrl else "#a3a29b"}},
                             error_y={"type": "data", "array": (sub.hi - sub.lo) / 2, "thickness": 8, "width": 0,
                                      "color": "rgba(110,109,102,0.35)" if not ctrl else "rgba(163,162,155,0.35)"},
                             customdata=sub.check, hovertemplate="%{customdata}: n %{x}<extra>range</extra>"))
for k, s in enumerate(dict.fromkeys(t2t.series)):
    sub = t2t[(t2t.series == s) & (t2t["axis.load"] != "none")].sort_values("n")
    # a small x offset per series, so that equal values do not hide each other (the hover gives the true n)
    fig.add_trace(go.Scatter(x=sub.n + (k - 1) * 1.5, y=sub["p50 - none"], mode="markers", name=s,
                             customdata=list(zip(sub.point, sub.n)),
                             marker={"size": 10, "color": nbfig.color(k),
                                     "symbol": ["circle" if l.startswith("t") else "diamond" for l in sub["axis.load"]]},
                             hovertemplate="%{customdata[0]}: n %{customdata[1]}, %{y} cycles<extra>" + s + "</extra>"))
xs = [5, 125]
# c = 20..25 cycles per node (study.md); c x n - 640: the basis of H1 c..g (hypotheses.yaml)
for c, dash in ((20, "dash"), (25, "dot")):
    fig.add_trace(go.Scatter(x=xs, y=[c * x - 640 for x in xs], mode="lines", name=f"model c = {c}",
                             line={"color": "#6e6d66", "dash": dash, "width": 1}, hoverinfo="skip"))
nbfig.style(fig, "t2t p50 minus the p50 without load, against n (circle: timers, diamond: sleepers)",
            "n (timeouts before the slice deadline, experiment.md)", "p50 - p50 of none (cycles)").show()
display(lim.set_index("check"))
t2t.pivot_table(index="axis.load", columns="series", values="p50 - none", aggfunc="first", sort=False)

## H1 h: one slope for the three primitives

H1 h (manual): the p50 is linear in `n`, with the same slope `c` for
sem, mutex and msgq. The differences between the primitives stay
constant. The basis: the walk over the timeouts is in the scheduler
(slice reset), not in the primitive. The first table gives the p50 and
the differences to sem. The second table gives the least-squares slope
and the intercept of the p50 over `n`, for the loaded points with time
slicing.

In [ ]:
import numpy as np

cube = t2t[t2t["axis.slicing"] == "default"]
p = cube.pivot_table(index="axis.load", columns="axis.primitive", values="p50", aggfunc="first", sort=False)
for prim in [c for c in p.columns if c != "sem"]:
    p[f"{prim} - sem"] = p[prim] - p["sem"]
p.insert(0, "n", [N.get(l) for l in p.index])
display(p)
fit = []
for prim, sub in cube[cube["axis.load"] != "none"].groupby("axis.primitive", sort=False):
    slope, icept = np.polyfit(sub.n, sub.p50, 1)
    fit.append({"primitive": prim, "slope (cycles per n)": round(slope, 2), "intercept": round(icept, 1),
                "points": len(sub)})
pd.DataFrame(fit).set_index("primitive")

## H2: the tail over the load

H2 expects that the `t2t` p99.99 grows with the number of timers (H2 a:
none, t16, t64, t256) and with the number of sleepers (H2 b: none, sl8,
sl64). The basis: the SysTick ISR runs the expiry functions of the
timers, and each sleep of a sleeper adds a timeout that expires in the
SysTick ISR. The colour is the primitive, the symbol is the
statistic.

In [ ]:
order = list(N)
cube = cube.assign(load=pd.Categorical(cube["axis.load"], order, ordered=True)).sort_values("load")
nbfig.dots(cube.assign(metric="t2t"), "t2t tail per load (time slicing on)", x="axis.load", group="axis.primitive",
           stats=("p99.99", "max")).show()
cube.pivot_table(index="axis.load", columns="axis.primitive", values=["p99", "p99.99", "max"], aggfunc="first",
                 sort=False)

In [ ]:
nbfig.ccdf({p: h for p, h in nbfig.hists_of(inst, "thread-load-1m").items()
            if p.startswith("sem-") and not p.endswith("noslice")},
           "thread-load-1m: CCDF of sem (time slicing on)", ["t2t", "t2t_ret", "isr2thr"]).show()